In [ ]:
"""Fusion LoRA SFT+DPO : export autonome de Qwen3 pour vLLM."""

import json
from pathlib import Path

import torch
from peft import PeftModel
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
)


# Localise la racine du projet depuis le notebook.
# Le dossier des datasets sert de repère indépendant du chemin courant.
def find_project_root(start: Path | None = None) -> Path:
    """Retrouve la racine du projet."""
    path = (start or Path.cwd()).resolve()

    for root in (path, *path.parents):
        if (root / "data/raw/datasets").is_dir():
            return root

    raise FileNotFoundError("Racine du projet introuvable.")


# Configuration : base exacte, adapter final et destination.
# Le DPO doit prolonger le SFT dans le même adapter LoRA.
PROJECT_ROOT = find_project_root()
RUN_STAMP = "20261009-154320"

BASE_MODEL = "Qwen/Qwen3-1.7B"

ADAPTER_PATH = (
    PROJECT_ROOT / "artifacts/finetuning_sft_dpo/models"
    / RUN_STAMP / "dpo_adapter"
)

OUTPUT_PATH = (
    PROJECT_ROOT / "artifacts/deployment"
    / f"qwen3-triage-{RUN_STAMP}"
)


# Reconstruit le modèle complet à partir de Qwen3 et du LoRA final.
# Fusionne les poids en BF16 sans conserver la dépendance PEFT.
def merge_lora() -> None:
    """Exporte Qwen3 fusionné au format Hugging Face."""

    # Vérifie l'existence et le type de l'adapter.
    # Affiche également la base déclarée pendant l'entraînement.
    config_path = ADAPTER_PATH / "adapter_config.json"

    if not config_path.is_file():
        raise FileNotFoundError(config_path)

    adapter_config = json.loads(config_path.read_text(encoding="utf-8"))

    if adapter_config.get("peft_type") != "LORA":
        raise ValueError("Adapter LoRA attendu.")

    declared_base = adapter_config.get("base_model_name_or_path")

    print(f"Base configurée : {BASE_MODEL}")
    print(f"Base adapter    : {declared_base}")

    if declared_base and declared_base != BASE_MODEL:
        print("⚠ Vérifier la correspondance des poids de base.")

    # Recharge le modèle original en BF16 sur CPU.
    # Évite une fusion directement sur les poids QLoRA 4-bit.
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.bfloat16,
        device_map="cpu",
        low_cpu_mem_usage=True,
    )

    # Recharge le tokenizer enregistré après le DPO.
    # Contrôle les capacités des embeddings d'entrée et de sortie.
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

    input_size = model.get_input_embeddings().num_embeddings
    output_size = model.get_output_embeddings().weight.shape[0]

    if len(tokenizer) > min(input_size, output_size):
        raise ValueError(
            f"Tokenizer incompatible : {len(tokenizer)} tokens, "
            f"embeddings {input_size}/{output_size}."
        )

    # Vérifie les identifiants des tokens de fin de génération.
    # Empêche l'export d'une configuration EOS invalide.
    eos_tokens = ("<|im_end|>", "<|endoftext|>")
    eos_ids = [tokenizer.convert_tokens_to_ids(token) for token in eos_tokens]

    if any(
        token_id is None
        or token_id < 0
        or token_id >= len(tokenizer)
        or token_id == tokenizer.unk_token_id
        for token_id in eos_ids
    ):
        raise ValueError("Tokens EOS introuvables ou invalides.")

    if tokenizer.pad_token_id is None:
        raise ValueError("PAD token non défini.")

    # Charge les matrices LoRA issues du fine-tuning.
    # Le DPO final doit déjà intégrer les modifications SFT.
    model = PeftModel.from_pretrained(
        model,
        str(ADAPTER_PATH),
        is_trainable=False,
    )

    # Fusionne les matrices LoRA avec les poids du modèle.
    # Supprime les couches PEFT devenues inutiles.
    model = model.merge_and_unload(safe_merge=True)

    # Configure les tokens d'arrêt et de remplissage.
    # Sauvegarde explicitement la configuration de génération.
    model.generation_config.eos_token_id = eos_ids
    model.generation_config.pad_token_id = tokenizer.pad_token_id

    # Exporte les poids BF16 au format Safetensors.
    # Conserve le tokenizer exact utilisé pendant l'entraînement.
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        OUTPUT_PATH,
        safe_serialization=True,
        max_shard_size="2GB",
    )

    tokenizer.save_pretrained(OUTPUT_PATH)
    model.generation_config.save_pretrained(OUTPUT_PATH)

    # Conserve les informations nécessaires à la traçabilité.
    # Ne modifie pas les fichiers standards du modèle Hugging Face.
    metadata = {
        "base_model": BASE_MODEL,
        "adapter_path": str(ADAPTER_PATH),
        "run_stamp": RUN_STAMP,
        "dtype": "bfloat16",
        "format": "safetensors",
        "merge_method": "merge_and_unload",
        "tokenizer_size": len(tokenizer),
        "input_embeddings": input_size,
        "output_embeddings": output_size,
        "eos_token_ids": eos_ids,
        "pad_token_id": tokenizer.pad_token_id,
    }

    (OUTPUT_PATH / "merge_metadata.json").write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    # Vérifie la présence des fichiers nécessaires à l'inférence.
    # Confirme l'absence de configuration PEFT dans l'export.
    required = (
        "config.json",
        "tokenizer_config.json",
        "generation_config.json",
        "merge_metadata.json",
    )

    if (
        any(not (OUTPUT_PATH / name).is_file() for name in required)
        or not list(OUTPUT_PATH.glob("*.safetensors"))
        or (OUTPUT_PATH / "adapter_config.json").exists()
    ):
        raise RuntimeError("Export autonome incomplet.")

    # Recharge les configurations et le tokenizer depuis l'export.
    # Vérifie leur cohérence sans recharger tous les poids en RAM.
    exported_config = AutoConfig.from_pretrained(OUTPUT_PATH)
    exported_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_PATH)
    exported_generation = GenerationConfig.from_pretrained(OUTPUT_PATH)

    if (
        len(exported_tokenizer) != len(tokenizer)
        or exported_config.vocab_size < len(exported_tokenizer)
        or exported_generation.eos_token_id != eos_ids
        or exported_generation.pad_token_id != tokenizer.pad_token_id
    ):
        raise RuntimeError("Incohérence détectée après export.")

    print(
        "\n✓ Fusion LoRA terminée\n"
        f"✓ Base       : {BASE_MODEL}\n"
        f"✓ Adapter    : {ADAPTER_PATH}\n"
        f"✓ Tokenizer  : {len(tokenizer):,} tokens\n"
        f"✓ Embeddings : {input_size:,} / {output_size:,} lignes\n"
        f"✓ EOS        : {eos_ids}\n"
        f"✓ PAD        : {tokenizer.pad_token_id}\n"
        f"✓ Format     : BF16 / Safetensors\n"
        f"✓ Export     : {OUTPUT_PATH}\n"
        "✓ Contrôles  : OK"
    )


# Exécute la fusion depuis le notebook ou un script Python.
merge_lora()

W1011 14:35:11.305000 25156 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

c:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gonza\.cache\huggingface\hub\models--Qwen--Qwen3-1.7B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

c:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gonza\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

✓ Base       : Qwen/Qwen3-1.7B
✓ Adapter    : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\finetuning_sft_dpo\models\20261009-154320\dpo_adapter
✓ Tokenizer  : 151,669 tokens
✓ Embeddings : 151,936 lignes
✓ Export     : C:\Users\gonza\Openclassroom\projet_14\medical_triage_agent\artifacts\deployment\qwen3-triage-20261009-154320
